In [1]:
def Auto_DeepLearning_pytorch(vy,vx,model_list,timestep=None,dilations=None,ifmulti_scale='no',num_channels=None,test_size=0.2,valid_size=0.1,k_fold=None,task_mode='regression',if_best_mode='no',modelpath=None,ifrandom_split='yes',cov_padding='same',cov_strides=1,pooling_strides=2,dropout=0.0,residual_deep=1,residual_activation='relu',residual_norm='batchnormalization',vit_patch_size=None,vit_dim_feedforward=16,vit_deep=1,vit_num_heads=1,swin_deep=[2,2],swin_num_heads=[1,1],cbam_reduction=16,transformer_encoder_deep=1,transformer_num_heads=2,transformer_dim_feedforward=256,tcn_kernel_size=2,nlp_units=64,nlp_num_layers=1,activation='relu',if_print_model='yes',loss_function='default',optimizer='SGD',metrics='default',learning_rate=0.01,epochs=2000,batch_size=20,if_early_stopping=None,ifheatmap='yes',ifweight='yes',ifmute='no',ifsave='no',savepath=None,device='cpu'):
    from torch.nn import Module,BatchNorm1d,BatchNorm2d,LayerNorm,Conv2d,MaxPool2d,AvgPool2d,RNN,LSTM,GRU,Embedding,TransformerEncoderLayer,TransformerEncoder,Dropout,LeakyReLU,ReLU,PReLU,Sigmoid,Tanh,ELU,Softmax,Linear,Flatten,ZeroPad2d
    from torch.optim import Adam,SGD,NAdam
    import torch
    from torch import nn
    import torchmetrics
    #from pytorch_tcn import TCN
    #from vit_pytorch import ViT
    #from timm.models.swin_transformer import SwinTransformer
    import math
    from sklearn.model_selection import train_test_split
    from sklearn.model_selection import KFold
    import numpy as np
    from scipy.stats import pearsonr
    import os
    from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score
    import sklearn
    import copy
    import shap
    import datetime
    import warnings
    from tqdm import tqdm
    warnings.filterwarnings('ignore')

    vx_original_ndim = vx.ndim
    
    # ============================================================
    # Device selection
    # 支持：
    #   device='cpu'
    #   device='gpu' 或 device='cuda'   -> cuda:0
    #   device='cuda:0'、'cuda:1'、...
    #   device=0、1、2、...
    #   device=torch.device(...)
    # ============================================================
    if isinstance(device, torch.device):
        devices = device

    elif isinstance(device, (int, np.integer)):
        devices = torch.device(f'cuda:{int(device)}')

    elif isinstance(device, str):
        device_lower = device.strip().lower()

        if device_lower == 'cpu':
            devices = torch.device('cpu')

        elif device_lower in ('gpu', 'cuda'):
            devices = torch.device('cuda:0')

        elif device_lower.startswith('cuda:'):
            gpu_index_text = device_lower.split(':', 1)[1]

            if not gpu_index_text.isdigit():
                raise ValueError(
                    f"device={device!r} 格式错误。"
                    "GPU设备应写成 'cuda:0'、'cuda:1' 等。"
                )

            devices = torch.device(
                f"cuda:{int(gpu_index_text)}"
            )

        else:
            raise ValueError(
                f"不支持的 device={device!r}。"
                "请使用 'cpu'、'gpu'、'cuda'、"
                "'cuda:0'、'cuda:1' 等，"
                "或者直接传入整数GPU编号。"
            )

    else:
        raise TypeError(
            "device 必须是 str、int、numpy整数或 torch.device，"
            f"当前类型为 {type(device).__name__}。"
        )

    # CUDA可用性和GPU编号检查
    if devices.type == 'cuda':
        if not torch.cuda.is_available():
            raise RuntimeError(
                f"指定了 {devices}，但当前 PyTorch 无法使用 CUDA。\n"
                f"PyTorch版本：{torch.__version__}\n"
                f"PyTorch内置CUDA版本：{torch.version.cuda}"
            )

        gpu_index = (
            0 if devices.index is None
            else int(devices.index)
        )

        visible_gpu_count = torch.cuda.device_count()

        if gpu_index < 0 or gpu_index >= visible_gpu_count:
            raise RuntimeError(
                f"指定了 cuda:{gpu_index}，但当前进程只识别到 "
                f"{visible_gpu_count} 张GPU。\n"
                f"CUDA_VISIBLE_DEVICES="
                f"{os.environ.get('CUDA_VISIBLE_DEVICES')}\n"
                "注意：设置 CUDA_VISIBLE_DEVICES 后，"
                "PyTorch会对当前进程可见的GPU重新从 cuda:0 编号。"
            )

        devices = torch.device(f'cuda:{gpu_index}')

        # 设置当前进程的默认CUDA设备。
        # 后续代码仍然统一使用 devices，不改变原有训练逻辑。
        torch.cuda.set_device(devices)

    print('=' * 70)
    print('实际使用设备：', devices)

    if devices.type == 'cuda':
        print(
            'GPU名称：',
            torch.cuda.get_device_name(devices)
        )
        print(
            '当前进程可见GPU数量：',
            torch.cuda.device_count()
        )
        print(
            'CUDA_VISIBLE_DEVICES：',
            os.environ.get('CUDA_VISIBLE_DEVICES')
        )

    print('=' * 70)
    if task_mode=='binary_classify' or task_mode=='multi_classify':
        def get_class_weights_from_labels(vy, device=None, ignore_index=None, num_classes=None):
            y = torch.as_tensor(vy, device=device)
        
            # 如果你标签存成 (...,1)，内部 squeeze 掉这个 1（不改原 vy）
            if y.ndim >= 1 and y.shape[-1] == 1:
                y = y.squeeze(-1)
        
            y = y.reshape(-1)
        
            if ignore_index is not None:
                y = y[y != ignore_index]
        
            if y.is_floating_point():
                y = y[~torch.isnan(y)]
        
            y = y.long()
        
            if num_classes is None:
                num_classes = int(y.max().item()) + 1
        
            cnt = torch.bincount(y, minlength=num_classes).float()
            w = cnt.sum() / (num_classes * cnt.clamp_min(1.0))  # N/(C*count_c)
            return w  # shape: (C,)
        class_weights = get_class_weights_from_labels(vy, device=devices)
    if task_mode=='regression':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='MSELoss':
                loss=torch.nn.MSELoss()
            elif loss_function=='L1Loss':
                loss=torch.nn.L1Loss
            elif loss_function=='PoissonNLLLoss':
                loss=torch.nn.PoissonNLLLoss()
            elif loss_function=='GaussianNLLLoss':
                loss=torch.nn.GaussianNLLLoss()
            elif loss_function=='KLDivLoss':
                loss=torch.nn.KLDivLoss()
            elif loss_function=='HuberLoss':
                loss=torch.nn.HuberLoss()
            elif loss_function=='SmoothL1Loss':
                loss=torch.nn.SmoothL1Loss()
            elif loss_function=='Pearsonr':
                class loss_pearsonr(nn.Module):
                    def __init__(self):
                        super().__init__()
    
                    def forward(self, y, x):
                        y_true_mean=torch.nanmean(y,dim=0,keepdim=True)
                        y_pred_mean=torch.nanmean(x,dim=0,keepdim=True)
                        cov=torch.nansum((y-y_true_mean)*(x-y_pred_mean),dim=0,keepdim=True)
                        y_true_v=torch.nansum(torch.square((y-y_true_mean)),dim=0,keepdim=True)
                        y_pred_v=torch.nansum(torch.square((x-y_pred_mean)),dim=0,keepdim=True)
                        y_true_v=torch.sqrt(y_true_v)
                        y_pred_v=torch.sqrt(y_pred_v)
                        pearson=cov/(y_true_v*y_pred_v)
                        return (1-pearson)**1.5
                loss=loss_pearsonr()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='MSELoss':
                metric=torch.nn.MSELoss()
            elif metrics=='L1Loss':
                metric=torch.nn.L1Loss
            elif metrics=='PoissonNLLLoss':
                metric=torch.nn.PoissonNLLLoss()
            elif metrics=='GaussianNLLLoss':
                metric=torch.nn.GaussianNLLLoss()
            elif metrics=='KLDivLoss':
                metric=torch.nn.KLDivLoss()
            elif metrics=='HuberLoss':
                metric=torch.nn.HuberLoss()
            elif metrics=='SmoothL1Loss':
                metric=torch.nn.SmoothL1Loss()
            elif metrics=='Pearsonr':
                class metric_pearsonr(nn.Module):
                    def __init__(self):
                        super().__init__()
    
                    def forward(self, y, x):
                        y_true_mean=torch.nanmean(y,dim=0,keepdim=True)
                        y_pred_mean=torch.nanmean(x,dim=0,keepdim=True)
                        cov=torch.nansum((y-y_true_mean)*(x-y_pred_mean),dim=0,keepdim=True)
                        y_true_v=torch.nansum(torch.square((y-y_true_mean)),dim=0,keepdim=True)
                        y_pred_v=torch.nansum(torch.square((x-y_pred_mean)),dim=0,keepdim=True)
                        y_true_v=torch.sqrt(y_true_v)
                        y_pred_v=torch.sqrt(y_pred_v)
                        pearson=cov/(y_true_v*y_pred_v)
                        return (1-pearson)**1.5
                metric=metric_pearsonr()
    elif task_mode=='binary_classify':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='BCELoss':
                w0, w1 = class_weights[0], class_weights[1]
                weight_map = torch.where(y_batch > 0.5, w1, w0)
                loss=torch.nn.BCELoss(weight=weight_map)
            elif loss_function=='BCEWithLogitsLoss':
                pos_w = (class_weights[1] / class_weights[0]).to(devices).view(1)
                loss=torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
            elif loss_function=='SoftMarginLoss':
                loss=torch.nn.SoftMarginLoss()
            elif loss_function=='MultiLabelSoftMarginLoss':
                loss=torch.nn.MultiLabelSoftMarginLoss()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='f1':
                metric=torchmetrics.F1Score(task="binary").to(devices)
            elif metrics=='accuracy':
                metric=torchmetrics.Accuracy(task="binary").to(devices)
            elif metrics=='precision':
                metric=torchmetrics.Precision(task="binary").to(devices)
            elif metrics=='recall':
                metric=torchmetrics.Recall(task="binary").to(devices)
            elif metrics=='BCELoss':
                metric=torch.nn.BCELoss()
            elif metrics=='BCEWithLogitsLoss':
                metric=torch.nn.BCEWithLogitsLoss()
            elif metrics=='SoftMarginLoss':
                metric=torch.nn.SoftMarginLoss()
            elif metrics=='MultiLabelSoftMarginLoss':
                metric=torch.nn.MultiLabelSoftMarginLoss()
    elif task_mode=='multi_classify':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                loss=torch.nn.CrossEntropyLoss(weight=class_weights)
            elif loss_function=='NLLLoss':
                loss=torch.nn.NLLLoss(weight=class_weights)
            elif loss_function=='TripletMarginLoss':
                loss=torch.nn.TripletMarginLoss()
            elif loss_function=='KLDivergence':
                loss=torch.nn.KLDivergence()
            elif loss_function=='HingeEmbeddingLoss':
                loss=torch.nn.HingeEmbeddingLoss()
            elif loss_function=='MultiLabelMarginLoss':
                loss=torch.nn.MultiLabelMarginLoss()
            elif loss_function=='TripletMarginWithDistanceLoss':
                loss=torch.nn.TripletMarginWithDistanceLoss()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                metric=torchmetrics.Accuracy(task="multiclass",num_classes = int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='recall' :
                metric=torchmetrics.Recall(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='precision' :
                metric=torchmetrics.Precision(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='f1' :
                metric=torchmetrics.F1Score(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='CrossEntropyLoss':
                metric=torch.nn.CrossEntropyLoss()
            elif metrics=='NLLLoss':
                metric=torch.nn.NLLLoss()
            elif metrics=='TripletMarginLoss':
                metric=torch.nn.TripletMarginLoss()
            elif metrics=='KLDivergence':
                metric=torch.nn.KLDivergence()
            elif metrics=='HingeEmbeddingLoss':
                metric=torch.nn.HingeEmbeddingLoss()
            elif metrics=='MultiLabelMarginLoss':
                metric=torch.nn.MultiLabelMarginLoss()
            elif metrics=='TripletMarginWithDistanceLoss':
                metric=torch.nn.TripletMarginWithDistanceLoss()
    heatmap=0
    weights=0
    model=0
    predicty=0
    testy=0
    r=0
    p=0
    
    class EarlyStopping:
        def __init__(self, patience, delta=0):
            self.patience = patience
            self.counter = 0
            self.best_score = None
            self.early_stop = False
            self.val_loss_min = np.Inf
            self.delta = delta
            self.best_model_state_dict = None
    
        def __call__(self, val_loss, model):
            # 1. NaN 保护：一旦验证 loss 是 NaN，就停止训练，后面外层用 load_best_checkpoint 恢复
            if np.isnan(val_loss):
                print("Validation loss is NaN. Stopping early and will restore best model.")
                self.early_stop = True
                return
    
            score = -val_loss
    
            # 2. 正常 early stopping 逻辑
            if self.best_score is None:
                self.best_score = score
                self.save_checkpoint(val_loss, model)
            elif score < self.best_score + self.delta:
                # 没有提升
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                # 有提升，更新 best
                self.best_score = score
                self.save_checkpoint(val_loss, model)
                self.counter = 0
    
        def save_checkpoint(self, val_loss, model):
            # 这里仍然是 clone 当前最优模型
            self.best_model_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            self.val_loss_min = val_loss
    
        def load_best_checkpoint(self, model):
            # 新增：一行恢复 best 模型
            if self.best_model_state_dict is not None:
                model.load_state_dict(self.best_model_state_dict)
            else:
                print("Warning: load_best_checkpoint called but no best_model_state_dict was saved.")
    class ShapWrapperCombinedInput(nn.Module):
        def __init__(self, original_model, x_sample_shape, pos_sample_shape):
            super().__init__()
            self.original_model = original_model
            self.x_sample_shape = x_sample_shape
            self.pos_sample_shape = pos_sample_shape
            self.x_flat_size = np.prod(x_sample_shape)
            self.pos_flat_size = np.prod(pos_sample_shape)
            self.total_flat_size = self.x_flat_size + self.pos_flat_size
    
        def forward(self, combined_input_float):
            if combined_input_float.dim() == 1:
                combined_input_float = combined_input_float.unsqueeze(0)
            x_flat = combined_input_float[:, :self.x_flat_size]
            pos_flat = combined_input_float[:, self.x_flat_size:]
            x_reshaped = x_flat.reshape(-1, *self.x_sample_shape)
            position_indices_int = pos_flat.long()
            return self.original_model(x_reshaped, position_indices_int)

    
    if vy.ndim==1:
        vy=vy.reshape(vy.shape[0],1)
    if vx.ndim==2 and timestep!=None:
        vxs=np.zeros((vx.shape[0]-timestep+1,timestep,vx.shape[1]))
    elif vx.ndim==4 and timestep!=None:
        vxs=np.zeros((vx.shape[0]-timestep+1,timestep,vx.shape[1],vx.shape[2],vx.shape[3]))
    elif timestep==None:
        vxs=vx
        vys=vy
    if timestep!=None:
        for i in range(vx.shape[0]-timestep+1):
            vxs[i,:,:]=vx[i:i+timestep,:]
        vys=vy[timestep-1:]
    if vx.ndim==4 and timestep!=None:
        vxs=vxs.transpose(0,1,4,2,3)
    if ifrandom_split=='yes':
        trainy,testy,trainx,testx = train_test_split(vys,vxs,test_size=test_size,random_state=25)
    elif ifrandom_split=='no':
        index=int((1-test_size)*vy.shape[0])
        trainy=vys[:index,]
        testy=vys[index:,]
        trainx=vxs[:index,]
        testx=vxs[index:,]
    elif ifrandom_split=='all_train' or ifrandom_split=='all_test' or ifrandom_split=='just_model':
        trainy=vys
        testy=vys
        trainx=vxs
        testx=vxs
    if timestep!=None:
        input_channel=trainx.shape[2]
        train_position=np.zeros((trainx.shape[0],trainx.shape[1]))
        test_position=np.zeros((testx.shape[0],testx.shape[1]))
        for i in range(trainx.shape[0]):
            train_position[i,:]=np.arange(0,timestep,1)
        for i in range(testx.shape[0]):
            test_position[i,:]=np.arange(0,timestep,1)
    else:
        input_channel=trainx.shape[1]
        train_position=np.zeros((trainx.shape[0],6))
        test_position=np.zeros((testx.shape[0],6))
        for i in range(trainx.shape[0]):
            train_position[i,:]=np.arange(0,6,1)
        for i in range(testx.shape[0]):
            test_position[i,:]=np.arange(0,6,1)
    class TimeDistributed(nn.Module):
        def __init__(self, module, batch_first=False):
            super(TimeDistributed, self).__init__()
            self.module = module
            self.batch_first = batch_first
    
        def forward(self, x):
            if len(x.size()) <= 2:
                return self.module(x)
    
            if self.batch_first:
                batch_size, seq_len = x.shape[0], x.shape[1]
            else:
                seq_len, batch_size = x.shape[0], x.shape[1]
    
            x_reshaped = x.reshape(batch_size * seq_len, *x.shape[2:])
            y = self.module(x_reshaped)
    
            if self.batch_first:
                y = y.reshape(batch_size, seq_len, *y.shape[1:])
            else:
                y = y.reshape(seq_len, batch_size, *y.shape[1:])
            return y
    def find_string_in_nested_list(target_string, nested_list):
        for item in nested_list:
            if isinstance(item, list):
                if find_string_in_nested_list(target_string, item):
                    return True
            elif isinstance(item, str):
                if item == target_string:
                    return True
        return False
    class Model(nn.Module):
        def __init__(self,model_list,timestep,dilations,ifmulti_scale,num_channels,trainx,cov_padding,cov_strides,pooling_strides,dropout,residual_deep,residual_activation,residual_norm,vit_patch_size,vit_dim_feedforward,vit_deep,vit_num_heads,swin_deep,swin_num_heads,cbam_reduction,transformer_encoder_deep,transformer_num_heads,transformer_dim_feedforward,tcn_kernel_size,nlp_units,nlp_num_layers,activation):
            super(Model,self).__init__()
            exec('from torch import nn', globals(), self.__dict__)
            exec('from torch.nn import Module,BatchNorm1d,BatchNorm2d,LayerNorm,Conv2d,MaxPool2d,AvgPool2d,RNN,LSTM,GRU,Embedding,TransformerEncoderLayer,TransformerEncoder,Dropout,LeakyReLU,ReLU,PReLU,Sigmoid,Tanh,ELU,Softmax,Linear,Flatten,ZeroPad2d,AdaptiveMaxPool2d,AdaptiveAvgPool2d', globals(), self.__dict__)
            exec('import numpy as np', globals(), self.__dict__)
            exec('import torch', globals(), self.__dict__)
            #exec('from pytorch_tcn import TCN', globals(), self.__dict__)
            #exec('from vit_pytorch import ViT', globals(), self.__dict__)
            #exec('from timm.models.swin_transformer import SwinTransformer', globals(), self.__dict__)
            exec('import math', globals(), self.__dict__)
            exec('import torch', globals(), self.__dict__)
            exec('import torch.nn.functional as F', globals(), self.__dict__)
            
            self.model_list=model_list
            self.timestep=timestep
            self.dilations=dilations
            self.ifmulti_scale=ifmulti_scale
            self.num_channels=num_channels
            self.trainx=trainx
            self.cov_padding=cov_padding
            self.cov_strides=cov_strides
            self.pooling_strides=pooling_strides
            self.dropout=dropout
            self.residual_deep=residual_deep
            self.residual_activation=residual_activation
            self.residual_norm=residual_norm
            self.vit_patch_size=vit_patch_size
            self.vit_dim_feedforward=vit_dim_feedforward
            self.vit_deep=vit_deep
            self.vit_num_heads=vit_num_heads
            self.swin_deep=swin_deep
            self.swin_num_heads=swin_num_heads
            self.cbam_reduction=cbam_reduction
            self.transformer_encoder_deep=transformer_encoder_deep
            self.transformer_num_heads=transformer_num_heads
            self.transformer_dim_feedforward=transformer_dim_feedforward
            self.tcn_kernel_size=tcn_kernel_size
            self.nlp_units=nlp_units
            self.nlp_num_layers=nlp_num_layers
            self.activation=activation
            self.__dict__['self']=self
            if trainx.ndim>3 and timestep!=None:
                self.hight=self.trainx.shape[3]
                self.weight=self.trainx.shape[4]
            elif trainx.ndim>3 and timestep==None:
                self.hight=self.trainx.shape[2]
                self.weight=self.trainx.shape[3]
            if timestep!=None:
                exec('in_channels=self.trainx.shape[2]', globals(), self.__dict__)
            elif timestep==None:
                exec('in_channels=self.trainx.shape[1]', globals(), self.__dict__)
            self.__dict__['TimeDistributed']=TimeDistributed
            if ['flatten'] in model_list:
                self.flattened=False
            else:
                self.flattened=True
            if ifmulti_scale=='yes':
                exec('self.dense_input_len=0', globals(), self.__dict__)
                
            if find_string_in_nested_list('vit', self.model_list) or find_string_in_nested_list('swin', self.model_list):
                def find_best_common_divisor(a, b):
                    def get_common_divisors(a, b):
                        """获取 a 和 b 的所有公约数"""
                        def gcd(x, y):
                            while y:
                                x, y = y, x % y
                            return x
                        
                        common_gcd = gcd(a, b)
                        divisors = []
                        for i in range(2, int(common_gcd**0.5) + 1):
                            if common_gcd % i == 0:
                                divisors.append(i)
                                if i != common_gcd // i:
                                    divisors.append(common_gcd // i)
                        divisors.append(common_gcd)
                        return sorted(divisors)
                
                    def find_valid_divisor(a, b):
                        """寻找满足条件的公约数"""
                        divisors = get_common_divisors(a, b)
                        best_divisor = None
                        min_value = float('inf')
                        
                        for d in divisors:
                            a_div, b_div = a // d, b // d
                            if d == 1 or a_div == 1 or b_div == 1:
                                continue
                            if a_div * b_div > 16:
                                value = np.abs((a_div - d)) + np.abs((b_div - d))
                                if value < min_value:
                                    min_value = value
                                    best_divisor = d
                        return best_divisor
                
                    def adjust_values(a, b):
                        """顺序尝试增加 a、b 或轮流增加 a 和 b"""
                        steps = 0
                        while True:
                            steps += 1
                            
                            # 1. 增加 a
                            new_a = a + steps
                            divisor = find_valid_divisor(new_a, b)
                            if divisor:
                                return new_a, b, divisor
                            
                            # 2. 增加 b
                            new_b = b + steps
                            divisor = find_valid_divisor(a, new_b)
                            if divisor:
                                return a, new_b, divisor
                            
                            # 3. 先 a 后 b
                            if (steps % 2) == 1:
                                new_a, new_b = a + steps, b + steps-1
                            else:
                                new_a, new_b = a + steps, b + steps
                            divisor = find_valid_divisor(new_a, new_b)
                            if divisor:
                                return new_a, new_b, divisor
                
                            # 4. 先 B 后 A
                            if (steps % 2) == 1:
                                new_a, new_b = a + steps-1, b + steps
                            else:
                                new_a, new_b = a + steps, b + steps
                            divisor = find_valid_divisor(new_a, new_b)
                            if divisor:
                                return new_a, new_b, divisor
                    divisor = find_valid_divisor(a, b)
                    if divisor:
                        return a, b, divisor
                    else:
                        new_a, new_b, new_divisor = adjust_values(a, b)
                        return new_a, new_b, new_divisor
                def find_new_a_b(a, b, divisor):
                    """
                    找到新的 a 和 b，使得 divisor 成为它们的公约数。
                    """
                    def is_common_divisor(x, y, d):
                        # 检查 d 是否为 x 和 y 的公约数
                        return x % d == 0 and y % d == 0
                    if is_common_divisor(a, b, divisor):
                            return a, b
                    # 初始化三个并行步骤的 a 和 b
                    a1, b1 = a, b  # 只增加 a
                    a2, b2 = a, b  # 只增加 b
                    a3, b3 = a, b  # a 和 b 轮流增加
                    a4, b4 = a, b  # a 和 b 轮流增加
                
                    # 循环直到找到满足条件的 a 和 b
                    while True:
                        # 步骤 1：只增加 a
                        a1 += 1
                        if is_common_divisor(a1, b1, divisor):
                            return a1, b1
                
                        # 步骤 2：只增加 b
                        b2 += 1
                        if is_common_divisor(a2, b2, divisor):
                            return a2, b2
                
                        # 步骤 3：a 和 b 轮流增加
                        if (a3 + b3) % 2 == 0:  # 偶数轮，增加 a
                            a3 += 1
                        else:  # 奇数轮，增加 b
                            b3 += 1
                        if is_common_divisor(a3, b3, divisor):
                            return a3, b3
                        if (a4 + b4) % 2 == 0:  # 偶数轮，增加 b
                            b4 += 1
                        else:  # 奇数轮，增加 a
                            a4 += 1
                        if is_common_divisor(a4, b4, divisor):
                            return a4, b4
            for i in range(len(self.model_list)):
                self.__dict__['i']=i
                if self.model_list[i][0] == 'cov' and timestep!=None:
                    if self.cov_padding=='valid':
                        self.hight=1+math.floor((self.hight-self.model_list[i][1][1])/self.cov_strides)
                        self.weight=1+math.floor((self.weight-self.model_list[i][1][2])/self.cov_strides)
                        if self.hight < 1 or self.weight < 1:
                            print('卷积层数过多')
                            break
                    exec('self.conv'+str(i+1)+'=TimeDistributed(Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding), batch_first=True)', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1][0]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'cov' and timestep==None:
                    if self.cov_padding=='valid':
                        self.hight=1+math.floor((self.hight-self.model_list[i][1][1])/self.cov_strides)
                        self.weight=1+math.floor((self.weight-self.model_list[i][1][2])/self.cov_strides)
                        if self.hight < 1 or self.weight < 1:
                            print('卷积层数过多')
                            break
                    exec('self.conv'+str(i+1)+'=Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding)', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1][0]', globals(), self.__dict__)                 
                elif self.model_list[i][0] == 'resnet' and timestep!=None:
                    if cov_padding=='same':
                        exec('self.pad'+str(i+1)+'=TimeDistributed(ZeroPad2d(padding=(math.floor(math.floor((self.trainx.shape[4]-(self.trainx.shape[4]/self.cov_strides)))/2),math.ceil(math.floor((self.trainx.shape[4]-(self.trainx.shape[4]/self.cov_strides)))/2),math.floor(math.floor((self.trainx.shape[3]-(self.trainx.shape[3]/self.cov_strides)))/2),math.ceil(math.floor((self.trainx.shape[3]-(self.trainx.shape[3]/self.cov_strides)))/2))), batch_first=True)', globals(), self.__dict__)
                        exec('self.conv'+str(i+1)+'=TimeDistributed(Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding), batch_first=True)', globals(), self.__dict__)
                    else:
                        exec('self.pad'+str(i+1)+'=TimeDistributed(ZeroPad2d(padding=(math.floor(math.floor((self.trainx.shape[4]-((self.trainx.shape[4]-self.model_list[i][1][1]+1)/self.cov_strides)))/2),math.ceil(math.floor((self.trainx.shape[4]-((self.trainx.shape[4]-self.model_list[i][1][1]+1)/self.cov_strides)))/2),math.floor(math.floor((self.trainx.shape[3]-((self.trainx.shape[3]-self.model_list[i][1][2]+1)/self.cov_strides)))/2),math.ceil(math.floor((self.trainx.shape[3]-((self.trainx.shape[3]-self.model_list[i][1][2]+1)/self.cov_strides)))/2))), batch_first=True)', globals(), self.__dict__)
                        exec('self.conv'+str(i+1)+'=TimeDistributed(Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding), batch_first=True)', globals(), self.__dict__)
                    exec('self.convs'+str(i+1)+'=TimeDistributed(Conv2d(in_channels,self.model_list[i][1][0],(1,1),stride=1,padding=self.cov_padding), batch_first=True)', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1][0]', globals(), self.__dict__)
                    for j in range(self.residual_deep):
                        if self.residual_norm=='batchnormalization':
                            exec('self.norm'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(BatchNorm2d(in_channels), batch_first=True)', globals(), self.__dict__)
                        elif self.residual_norm=='layernormalization':
                            exec('self.norm'+str(i+1)+'_'+str(j+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                        if self.activation=='elu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ELU()', globals(), self.__dict__)
                        elif self.activation=='leakyrelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=LeakyReLU()', globals(), self.__dict__)
                        elif self.activation=='prelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=PReLU()', globals(), self.__dict__)
                        elif self.activation=='relu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ReLU()', globals(), self.__dict__)
                        elif self.activation=='sigmoid':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Sigmoid()', globals(), self.__dict__)
                        elif self.activation=='tanh':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Tanh()', globals(), self.__dict__)
                        elif self.activation=='softmax':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Softmax()', globals(), self.__dict__)
                        if cov_padding=='same':
                            exec('self.pad'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(ZeroPad2d(padding=(math.floor(math.floor((self.weight-(self.weight/self.cov_strides)))/2),math.ceil(math.floor((self.weight-(self.weight/self.cov_strides)))/2),math.floor(math.floor((self.hight-(self.hight/self.cov_strides)))/2),math.ceil(math.floor((self.hight-(self.hight/self.cov_strides)))/2))), batch_first=True)', globals(), self.__dict__)
                            exec('self.conv'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding), batch_first=True)', globals(), self.__dict__)
                        else:
                            exec('self.pad'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(ZeroPad2d(padding=(math.floor(math.floor((self.weight-((self.weight-self.model_list[i][1][1]+1)/self.cov_strides)))/2),math.ceil(math.floor((self.weight-((self.weight-self.model_list[i][1][1]+1)/self.cov_strides)))/2),math.floor(math.floor((self.hight-((self.hight-self.model_list[i][1][2]+1)/self.cov_strides)))/2),math.ceil(math.floor((self.hight-((self.hight-self.model_list[i][1][2]+1)/self.cov_strides)))/2))), batch_first=True)', globals(), self.__dict__)
                            exec('self.conv'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding), batch_first=True)', globals(), self.__dict__)
                    exec('self.add'+str(i+1)+'=torch.add', globals(), self.__dict__)
                elif self.model_list[i][0] == 'resnet' and timestep==None:
                    if cov_padding=='same':
                        exec('self.pad'+str(i+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.trainx.shape[3]-(self.trainx.shape[3]/self.cov_strides)))/2),math.ceil(math.floor((self.trainx.shape[3]-(self.trainx.shape[3]/self.cov_strides)))/2),math.floor(math.floor((self.trainx.shape[2]-(self.trainx.shape[2]/self.cov_strides)))/2),math.ceil(math.floor((self.trainx.shape[2]-(self.trainx.shape[2]/self.cov_strides)))/2)))', globals(), self.__dict__)
                        exec('self.conv'+str(i+1)+'=Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding)', globals(), self.__dict__)
                    else:
                        exec('self.pad'+str(i+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.trainx.shape[3]-((self.trainx.shape[3]-self.model_list[i][1][1]+1)/self.cov_strides)))/2),math.ceil(math.floor((self.trainx.shape[3]-((self.trainx.shape[3]-self.model_list[i][1][1]+1)/self.cov_strides)))/2),math.floor(math.floor((self.trainx.shape[2]-((self.trainx.shape[2]-self.model_list[i][1][2]+1)/self.cov_strides)))/2),math.ceil(math.floor((self.trainx.shape[2]-((self.trainx.shape[2]-self.model_list[i][1][2]+1)/self.cov_strides)))/2)))', globals(), self.__dict__)
                        exec('self.conv'+str(i+1)+'=Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding)', globals(), self.__dict__)
                    exec('self.convs'+str(i+1)+'=Conv2d(in_channels,self.model_list[i][1][0],(1,1),stride=1,padding=self.cov_padding)', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1][0]', globals(), self.__dict__)
                    for j in range(self.residual_deep):
                        if self.residual_norm=='batchnormalization':
                            exec('self.norm'+str(i+1)+'_'+str(j+1)+'=BatchNorm2d(in_channels)', globals(), self.__dict__)
                        elif self.residual_norm=='layernormalization':
                            exec('self.norm'+str(i+1)+'_'+str(j+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                        if self.activation=='elu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ELU()', globals(), self.__dict__)
                        elif self.activation=='leakyrelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=LeakyReLU()', globals(), self.__dict__)
                        elif self.activation=='prelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=PReLU()', globals(), self.__dict__)
                        elif self.activation=='relu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ReLU()', globals(), self.__dict__)
                        elif self.activation=='sigmoid':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Sigmoid()', globals(), self.__dict__)
                        elif self.activation=='tanh':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Tanh()', globals(), self.__dict__)
                        elif self.activation=='softmax':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Softmax()', globals(), self.__dict__)
                        if cov_padding=='same':
                            exec('self.pad'+str(i+1)+'_'+str(j+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight-(self.weight/self.cov_strides)))/2),math.ceil(math.floor((self.weight-(self.weight/self.cov_strides)))/2),math.floor(math.floor((self.hight-(self.hight/self.cov_strides)))/2),math.ceil(math.floor((self.hight-(self.hight/self.cov_strides)))/2)))', globals(), self.__dict__)
                            exec('self.conv'+str(i+1)+'_'+str(j+1)+'=Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding)', globals(), self.__dict__)
                        else:
                            exec('self.pad'+str(i+1)+'_'+str(j+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight-((self.weight-self.model_list[i][1][1]+1)/self.cov_strides)))/2),math.ceil(math.floor((self.weight-((self.weight-self.model_list[i][1][1]+1)/self.cov_strides)))/2),math.floor(math.floor((self.hight-((self.hight-self.model_list[i][1][2]+1)/self.cov_strides)))/2),math.ceil(math.floor((self.hight-((self.hight-self.model_list[i][1][2]+1)/self.cov_strides)))/2)))', globals(), self.__dict__)
                            exec('self.conv'+str(i+1)+'_'+str(j+1)+'=Conv2d(in_channels,self.model_list[i][1][0],(self.model_list[i][1][1],self.model_list[i][1][2]),stride=self.cov_strides,padding=self.cov_padding)', globals(), self.__dict__)
                    exec('self.add'+str(i+1)+'=torch.add', globals(), self.__dict__)
                elif self.model_list[i][0] == 'vit' and timestep!=None:
                    if self.vit_patch_size==None:
                        self.new_hights,self.new_weights,self.vit_patch_size=find_best_common_divisor(self.hight,self.weight)
                    else:
                        self.new_hights,self.new_weights=find_new_a_b(self.hight,self.weight,self.vit_patch_size)
                    exec('self.vit'+str(i+1)+'=TimeDistributed(ViT(image_size = (self.new_hights,self.new_weights),patch_size = self.vit_patch_size,num_classes = (self.model_list[i][1]*self.hight*self.weight),dim = self.vit_dim_feedforward,depth = self.vit_deep,heads = self.vit_num_heads,mlp_dim = self.vit_dim_feedforward,channels=in_channels,dropout = self.dropout), batch_first=True)', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'swin' and timestep!=None:
                    if self.vit_patch_size==None:
                        self.new_hights,self.new_weights,self.vit_patch_size=find_best_common_divisor(self.hight,self.weight)
                    else:
                        self.new_hights,self.new_weights=find_new_a_b(self.hight,self.weight,self.vit_patch_size)
                    exec('self.swin'+str(i+1)+'=TimeDistributed(SwinTransformer(img_size = (self.new_hights,self.new_weights),patch_size = self.vit_patch_size,num_classes = (self.model_list[i][1]*self.hight*self.weight),window_size=self.model_list[i][2],embed_dim = 96,depths = self.swin_deep,num_heads = self.swin_num_heads,in_chans=in_channels,drop_rate= self.dropout), batch_first=True)', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'vit' and timestep==None:
                    if self.vit_patch_size==None:
                        self.new_hights,self.new_weights,self.vit_patch_size=find_best_common_divisor(self.hight,self.weight)
                    else:
                        self.new_hights,self.new_weights=find_new_a_b(self.hight,self.weight,self.vit_patch_size)
                    exec('self.vit'+str(i+1)+'=ViT(image_size = (self.new_hights,self.new_weights),patch_size = self.vit_patch_size,num_classes = (self.model_list[i][1]*self.hight*self.weight),dim = self.vit_dim_feedforward,depth = self.vit_deep,heads = self.vit_num_heads,mlp_dim = self.vit_dim_feedforward,channels=in_channels,dropout = self.dropout)', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'swin' and timestep==None:
                    if self.vit_patch_size==None:
                        self.new_hights,self.new_weights,self.vit_patch_size=find_best_common_divisor(self.hight,self.weight)
                    else:
                        self.new_hights,self.new_weights=find_new_a_b(self.hight,self.weight,self.vit_patch_size)
                    exec('self.swin'+str(i+1)+'=SwinTransformer(img_size = (self.new_hights,self.new_weights),patch_size = self.vit_patch_size,num_classes = (self.model_list[i][1]*self.hight*self.weight),window_size=self.model_list[i][2],embed_dim = 96,depths = self.swin_deep,num_heads = self.swin_num_heads,in_chans=in_channels,drop_rate= self.dropout)', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'cbam' and timestep!=None:
                    exec('self.cbam_max_pool'+str(i+1)+'=TimeDistributed(AdaptiveMaxPool2d(1), batch_first=True)', globals(), self.__dict__)
                    exec('self.cbam_ave_pool'+str(i+1)+'=TimeDistributed(AdaptiveAvgPool2d(1), batch_first=True)', globals(), self.__dict__)
                    exec('self.cbam_conv'+str(i+1)+'_1=TimeDistributed(Conv2d(in_channels,in_channels // cbam_reduction,1,bias=False), batch_first=True)', globals(), self.__dict__)
                    exec('self.cbam_act'+str(i+1)+'_1=ReLU()', globals(), self.__dict__)
                    exec('self.cbam_conv'+str(i+1)+'_2=TimeDistributed(Conv2d(in_channels // cbam_reduction,in_channels,1,bias=False), batch_first=True)', globals(), self.__dict__)
                    exec('self.cbam_act'+str(i+1)+'_2=Sigmoid()', globals(), self.__dict__)
                    exec('self.cbam_conv'+str(i+1)+'_3=TimeDistributed(Conv2d(2,1,kernel_size=7,padding=7 // 2,bias=False), batch_first=True)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'cbam' and timestep ==None:
                    exec('self.cbam_max_pool'+str(i+1)+'=AdaptiveMaxPool2d(1)', globals(), self.__dict__)
                    exec('self.cbam_ave_pool'+str(i+1)+'=AdaptiveAvgPool2d(1)', globals(), self.__dict__)
                    exec('self.cbam_conv'+str(i+1)+'_1=Conv2d(in_channels,in_channels // cbam_reduction,1,bias=False)', globals(), self.__dict__)
                    exec('self.cbam_act'+str(i+1)+'_1=ReLU()', globals(), self.__dict__)
                    exec('self.cbam_conv'+str(i+1)+'_2=Conv2d(in_channels // cbam_reduction,in_channels,1,bias=False)', globals(), self.__dict__)
                    exec('self.cbam_act'+str(i+1)+'_2=Sigmoid()', globals(), self.__dict__)
                    exec('self.cbam_conv'+str(i+1)+'_3=Conv2d(2,1,kernel_size=7,padding=7 // 2,bias=False)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'maxpooling' and timestep!=None:
                    if ifmulti_scale=='yes':
                        exec('self.fla'+str(i+1)+'=Flatten(start_dim=2, end_dim=-1)', globals(), self.__dict__)
                        exec('self.conc'+str(i+1)+'=torch.concatenate', globals(), self.__dict__)
                        exec('self.dense_input_len=self.dense_input_len+self.hight*self.weight*in_channels', globals(), self.__dict__)
                    self.hight=1+math.floor((self.hight-self.model_list[i][1][0])/self.pooling_strides)
                    self.weight=1+math.floor((self.weight-self.model_list[i][1][1])/self.pooling_strides)
                    if self.hight < 1 or self.weight < 1:
                        print('池化层数过多')
                        break
                    exec('self.pool'+str(i+1)+'=TimeDistributed(MaxPool2d((self.model_list[i][1][0],self.model_list[i][1][1]),stride=self.pooling_strides), batch_first=True)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'maxpooling' and timestep==None:
                    if ifmulti_scale=='yes':
                        exec('self.fla'+str(i+1)+'=Flatten()', globals(), self.__dict__)
                        exec('self.conc'+str(i+1)+'=torch.concatenate', globals(), self.__dict__)
                        exec('self.dense_input_len=self.dense_input_len+self.hight*self.weight*in_channels', globals(), self.__dict__)
                    self.hight=1+math.floor((self.hight-self.model_list[i][1][0])/self.pooling_strides)
                    self.weight=1+math.floor((self.weight-self.model_list[i][1][1])/self.pooling_strides)
                    if self.hight < 1 or self.weight < 1:
                        print('池化层数过多')
                        break
                    exec('self.pool'+str(i+1)+'=MaxPool2d((self.model_list[i][1][0],self.model_list[i][1][1]),stride=self.pooling_strides)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'avepooling' and timestep!=None:
                    if ifmulti_scale=='yes':
                        exec('self.fla'+str(i+1)+'=Flatten(start_dim=2, end_dim=-1)', globals(), self.__dict__)
                        exec('self.conc'+str(i+1)+'=torch.concatenate', globals(), self.__dict__)
                        exec('self.dense_input_len=self.dense_input_len+self.hight*self.weight*in_channels', globals(), self.__dict__)
                    self.hight=1+math.floor((self.hight-self.model_list[i][1][0])/self.pooling_strides)
                    self.weight=1+math.floor((self.weight-self.model_list[i][1][1])/self.pooling_strides)
                    if self.hight < 1 or self.weight < 1:
                        print('池化层数过多')
                        break
                    exec('self.pool'+str(i+1)+'=TimeDistributed(AvgPool2d((self.model_list[i][1][0],self.model_list[i][1][1]),stride=self.pooling_strides), batch_first=True)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'avepooling' and timestep==None:
                    if ifmulti_scale=='yes':
                        exec('self.fla'+str(i+1)+'=Flatten()', globals(), self.__dict__)
                        exec('self.conc'+str(i+1)+'=torch.concatenate', globals(), self.__dict__)
                        exec('self.dense_input_len=self.dense_input_len+self.hight*self.weight*in_channels', globals(), self.__dict__)
                    self.hight=1+math.floor((self.hight-self.model_list[i][1][0])/self.pooling_strides)
                    self.weight=1+math.floor((self.weight-self.model_list[i][1][1])/self.pooling_strides)
                    if self.hight < 1 or self.weight < 1:
                        print('池化层数过多')
                        break
                    exec('self.pool'+str(i+1)+'=AvgPool2d((self.model_list[i][1][0],self.model_list[i][1][1]),stride=self.pooling_strides)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'batchnormalization' and not self.flattened and timestep!=None:
                    exec('self.norm'+str(i+1)+'=TimeDistributed(BatchNorm2d(in_channels), batch_first=True)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'batchnormalization' and not self.flattened and timestep==None:
                    exec('self.norm'+str(i+1)+'=BatchNorm2d(in_channels)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'batchnormalization'and self.flattened:
                    exec('self.norm'+str(i+1)+'=BatchNorm1d(in_channels)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'layernormalization':
                    exec('self.norm'+str(i+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'activation':
                    if self.model_list[i][1]=='elu':
                        exec('self.act'+str(i+1)+'=ELU()', globals(), self.__dict__)
                    elif self.model_list[i][1]=='leakyrelu':
                        exec('self.act'+str(i+1)+'=LeakyReLU()', globals(), self.__dict__)
                    elif self.model_list[i][1]=='prelu':
                        exec('self.act'+str(i+1)+'=PReLU()', globals(), self.__dict__)
                    elif self.model_list[i][1]=='relu':
                        exec('self.act'+str(i+1)+'=ReLU()', globals(), self.__dict__)
                    elif self.model_list[i][1]=='sigmoid':
                        exec('self.act'+str(i+1)+'=Sigmoid()', globals(), self.__dict__)
                    elif self.model_list[i][1]=='tanh':
                        exec('self.act'+str(i+1)+'=Tanh()', globals(), self.__dict__)
                    elif self.model_list[i][1]=='softmax':
                        exec('self.act'+str(i+1)+'=Softmax()', globals(), self.__dict__)
                elif trainx.ndim>3 and self.model_list[i][0] == 'flatten' and timestep!=None:
                    exec('self.fla'+str(i+1)+'=Flatten(start_dim=2, end_dim=-1)', globals(), self.__dict__)
                    self.flattened=True
                    if ifmulti_scale=='yes':
                        exec('self.conc'+str(i+1)+'=torch.concatenate', globals(), self.__dict__)
                        exec('self.dense_input_len=self.dense_input_len+self.hight*self.weight*in_channels', globals(), self.__dict__)
                        exec('in_channels=self.dense_input_len', globals(), self.__dict__)
                    else:
                        exec('in_channels=self.hight*self.weight*in_channels', globals(), self.__dict__)
                elif trainx.ndim>3 and self.model_list[i][0] == 'flatten' and timestep==None:
                    exec('self.fla'+str(i+1)+'=Flatten()', globals(), self.__dict__)
                    self.flattened=True
                    if ifmulti_scale=='yes':
                        exec('self.conc'+str(i+1)+'=torch.concatenate', globals(), self.__dict__)
                        exec('self.dense_input_len=self.dense_input_len+self.hight*self.weight*in_channels', globals(), self.__dict__)
                        exec('in_channels=self.dense_input_len', globals(), self.__dict__)
                    else:
                        exec('in_channels=self.hight*self.weight*in_channels', globals(), self.__dict__)
                elif trainx.ndim<=3 and self.model_list[i][0] == 'flatten':
                    exec('self.fla'+str(i+1)+'=Flatten()', globals(), self.__dict__)
                    self.flattened=True
                elif self.model_list[i][0] =='fc':
                    exec('self.fc'+str(i+1)+'=Linear(in_channels,self.model_list[i][1])', globals(), self.__dict__)
                    exec('in_channels=self.model_list[i][1]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'dropout':
                    exec('self.drop'+str(i+1)+'=Dropout(self.model_list[i][1])', globals(), self.__dict__)
                elif self.model_list[i][0] == 'transformer':
                    exec('self.embedding'+str(i+1)+'=Embedding(self.timestep+1,in_channels)', globals(), self.__dict__)
                    exec('self.transformer_add'+str(i+1)+'=torch.add', globals(), self.__dict__)
                    exec('self.transformer'+str(i+1)+'=TransformerEncoder(TransformerEncoderLayer(in_channels,self.transformer_num_heads,dim_feedforward=self.transformer_dim_feedforward,dropout=self.dropout,activation=self.activation, batch_first=True),self.transformer_encoder_deep)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'tcn':
                    exec('self.tcn'+str(i+1)+'=TCN(in_channels,self.num_channels,kernel_size=self.tcn_kernel_size,dilations=self.dilations,dropout=self.dropout,activation=self.activation,input_shape="NLC")', globals(), self.__dict__)
                    exec('in_channels=self.num_channels[-1]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'lstm':
                    exec('self.lstm'+str(i+1)+'=LSTM(in_channels,self.nlp_units,num_layers=self.nlp_num_layers,batch_first=True,dropout=self.dropout)', globals(), self.__dict__)
                    exec('in_channels=self.nlp_units', globals(), self.__dict__)
                elif self.model_list[i][0] == 'rnn':
                    exec('self.rnn'+str(i+1)+'=RNN(in_channels,self.nlp_units,num_layers=self.nlp_num_layers,batch_first=True,dropout=self.dropout)', globals(), self.__dict__)
                    exec('in_channels=self.nlp_units', globals(), self.__dict__)
                elif self.model_list[i][0] == 'gru':
                    exec('self.gru'+str(i+1)+'=GRU(in_channels,self.nlp_units,num_layers=self.nlp_num_layers,batch_first=True,dropout=self.dropout)', globals(), self.__dict__)
                    exec('in_channels=self.nlp_units', globals(), self.__dict__)
        def forward(self, x, position=None):
            self.__dict__['x']=x
            self.__dict__['position']=position
            self.__dict__['self']=self
            try:
                exec('del model_conc', globals(), self.__dict__)
            except:
                pass
            if timestep!=None:
                def is_last_occurrence(target_string, full_nested_list, current_i):
        
                    if not isinstance(target_string, str):
                        return False
                
                    if not (0 <= current_i < len(full_nested_list)) or \
                       not isinstance(full_nested_list[current_i], list) or \
                       len(full_nested_list[current_i]) == 0:
                        return False
                
                    def search_in_segment(segment):
                        for item in segment:
                            if isinstance(item, list):
                                if search_in_segment(item):
                                    return True
                            elif isinstance(item, str):
                                if item == target_string:
                                    return True
                        return False
                
                    remaining_list_segment = full_nested_list[current_i + 1:]
                    return not search_in_segment(remaining_list_segment)
            if find_string_in_nested_list('vit', self.model_list) or find_string_in_nested_list('swin', self.model_list):
                def find_best_common_divisor(a, b):
                    def get_common_divisors(a, b):
                        """获取 a 和 b 的所有公约数"""
                        def gcd(x, y):
                            while y:
                                x, y = y, x % y
                            return x
                        
                        common_gcd = gcd(a, b)
                        divisors = []
                        for i in range(2, int(common_gcd**0.5) + 1):
                            if common_gcd % i == 0:
                                divisors.append(i)
                                if i != common_gcd // i:
                                    divisors.append(common_gcd // i)
                        divisors.append(common_gcd)
                        return sorted(divisors)
                
                    def find_valid_divisor(a, b):
                        """寻找满足条件的公约数"""
                        divisors = get_common_divisors(a, b)
                        best_divisor = None
                        min_value = float('inf')
                        
                        for d in divisors:
                            a_div, b_div = a // d, b // d
                            if d == 1 or a_div == 1 or b_div == 1:
                                continue
                            if a_div * b_div > 16:
                                value = np.abs((a_div - d)) + np.abs((b_div - d))
                                if value < min_value:
                                    min_value = value
                                    best_divisor = d
                        return best_divisor
                
                    def adjust_values(a, b):
                        """顺序尝试增加 a、b 或轮流增加 a 和 b"""
                        steps = 0
                        while True:
                            steps += 1
                            
                            # 1. 增加 a
                            new_a = a + steps
                            divisor = find_valid_divisor(new_a, b)
                            if divisor:
                                return new_a, b, divisor
                            
                            # 2. 增加 b
                            new_b = b + steps
                            divisor = find_valid_divisor(a, new_b)
                            if divisor:
                                return a, new_b, divisor
                            
                            # 3. 先 a 后 b
                            if (steps % 2) == 1:
                                new_a, new_b = a + steps, b + steps-1
                            else:
                                new_a, new_b = a + steps, b + steps
                            divisor = find_valid_divisor(new_a, new_b)
                            if divisor:
                                return new_a, new_b, divisor
                
                            # 4. 先 B 后 A
                            if (steps % 2) == 1:
                                new_a, new_b = a + steps-1, b + steps
                            else:
                                new_a, new_b = a + steps, b + steps
                            divisor = find_valid_divisor(new_a, new_b)
                            if divisor:
                                return new_a, new_b, divisor
                    divisor = find_valid_divisor(a, b)
                    if divisor:
                        return a, b, divisor
                    else:
                        new_a, new_b, new_divisor = adjust_values(a, b)
                        return new_a, new_b, new_divisor
                def find_new_a_b(a, b, divisor):
                    """
                    找到新的 a 和 b，使得 divisor 成为它们的公约数。
                    """
                    def is_common_divisor(x, y, d):
                        # 检查 d 是否为 x 和 y 的公约数
                        return x % d == 0 and y % d == 0
                    if is_common_divisor(a, b, divisor):
                            return a, b
                    # 初始化三个并行步骤的 a 和 b
                    a1, b1 = a, b  # 只增加 a
                    a2, b2 = a, b  # 只增加 b
                    a3, b3 = a, b  # a 和 b 轮流增加
                    a4, b4 = a, b  # a 和 b 轮流增加
                
                    # 循环直到找到满足条件的 a 和 b
                    while True:
                        # 步骤 1：只增加 a
                        a1 += 1
                        if is_common_divisor(a1, b1, divisor):
                            return a1, b1
                
                        # 步骤 2：只增加 b
                        b2 += 1
                        if is_common_divisor(a2, b2, divisor):
                            return a2, b2
                
                        # 步骤 3：a 和 b 轮流增加
                        if (a3 + b3) % 2 == 0:  # 偶数轮，增加 a
                            a3 += 1
                        else:  # 奇数轮，增加 b
                            b3 += 1
                        if is_common_divisor(a3, b3, divisor):
                            return a3, b3
                        if (a4 + b4) % 2 == 0:  # 偶数轮，增加 b
                            b4 += 1
                        else:  # 奇数轮，增加 a
                            a4 += 1
                        if is_common_divisor(a4, b4, divisor):
                            return a4, b4
            if timestep!=None:
                exec('in_channels=self.trainx.shape[2]', globals(), self.__dict__)
            elif timestep==None:
                exec('in_channels=self.trainx.shape[1]', globals(), self.__dict__)
            if trainx.ndim>3 and timestep!=None:
                self.hight=self.trainx.shape[3]
                self.weight=self.trainx.shape[4]
            elif trainx.ndim>3 and timestep==None:
                self.hight=self.trainx.shape[2]
                self.weight=self.trainx.shape[3]
            for i in range(len(self.model_list)):
                self.__dict__['i']=i
                if self.model_list[i][0] == 'cov':
                    if i==0:
                        exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if self.model_list[i-1][0]=='cov':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='resnet':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='vit':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='swin':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='cbam':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='activation' :
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='dropout' :
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                elif self.model_list[i][0] == 'resnet':
                    if i==0:
                        exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(x)', globals(), self.__dict__)
                        exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                        exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(x)', globals(), self.__dict__)
                        for j in range(self.residual_deep):
                            exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    else:
                        if self.model_list[i-1][0]=='conv':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='resnet':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='vit':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='swin':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='cbam':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.pad'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='activation' :
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='dropout' :
                            exec('model_conv'+str(i+1)+'=self.conv'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            exec('model_pad'+str(i+1)+'=self.pad'+str(i+1)+'(model_conv'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_convs'+str(i+1)+'=self.convs'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            for j in range(self.residual_deep):
                                exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+')', globals(), self.__dict__)
                                exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                                exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_add'+str(i+1)+'=self.add'+str(i+1)+'(model_convs'+str(i+1)+',model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                elif self.model_list[i][0] == 'vit':
                    if self.vit_patch_size==None:
                        self.new_hights,self.new_weights,self.vit_patch_size=find_best_common_divisor(self.hight,self.weight)
                    else:
                        self.new_hights,self.new_weights=find_new_a_b(self.hight,self.weight,self.vit_patch_size)
                    exec('in_channels=self.model_list[i][1]', globals(), self.__dict__)
                    if i==0:
                        exec('x = F.pad(x, (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                        exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if self.model_list[i-1][0]=='conv':
                            exec('model_conv'+str(i)+' = F.pad(model_conv'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='resnet':
                            exec('model_add'+str(i)+' = F.pad(model_add'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='vit':
                            exec('model_vit'+str(i)+' = F.pad(model_vit'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='swin':
                            exec('model_swin'+str(i)+' = F.pad(model_swin'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='cbam':
                            exec('model_cbam_out'+str(i)+'_2 = F.pad(model_cbam_out'+str(i)+'_2, (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                            exec('model_pool'+str(i)+' = F.pad(model_pool'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                            exec('model_norm'+str(i)+' = F.pad(model_norm'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='activation' :
                            exec('model_act'+str(i)+' = F.pad(model_act'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='dropout' :
                            exec('model_drop'+str(i)+' = F.pad(model_drop'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_vit'+str(i+1)+'=self.vit'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                    if self.timestep!=None:
                        exec('model_vit'+str(i+1)+'=model_vit'+str(i+1)+'.reshape(-1,self.timestep,in_channels,self.hight,self.weight)', globals(), self.__dict__)
                    else:
                        exec('model_vit'+str(i+1)+'=model_vit'+str(i+1)+'.reshape(-1,in_channels,self.hight,self.weight)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'swin':
                    if self.vit_patch_size==None:
                        self.new_hights,self.new_weights,self.vit_patch_size=find_best_common_divisor(self.hight,self.weight)
                    else:
                        self.new_hights,self.new_weights=find_new_a_b(self.hight,self.weight,self.vit_patch_size)
                    exec('in_channels=self.model_list[i][1]', globals(), self.__dict__)
                    if i==0:
                        exec('x = F.pad(x, (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                        exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if self.model_list[i-1][0]=='conv':
                            exec('model_conv'+str(i)+' = F.pad(model_conv'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='resnet':
                            exec('model_add'+str(i)+' = F.pad(model_add'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='vit':
                            exec('model_vit'+str(i)+' = F.pad(model_vit'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='swin':
                            exec('model_swin'+str(i)+' = F.pad(model_swin'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='cbam':
                            exec('model_cbam_out'+str(i)+'_2 = F.pad(model_cbam_out'+str(i)+'_2, (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                            exec('model_pool'+str(i)+' = F.pad(model_pool'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                            exec('model_norm'+str(i)+' = F.pad(model_norm'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='activation' :
                            exec('model_act'+str(i)+' = F.pad(model_act'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='dropout' :
                            exec('model_drop'+str(i)+' = F.pad(model_drop'+str(i)+', (int(np.floor((self.new_weights-self.weight)/2.0)),int(np.ceil((self.new_weights-self.weight)/2.0)),int(np.floor((self.new_hights-self.hight)/2.0)),int(np.ceil((self.new_hights-self.hight)/2.0))), mode="constant", value=0)', globals(), self.__dict__)
                            exec('model_swin'+str(i+1)+'=self.swin'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                    if self.timestep!=None:
                        exec('model_swin'+str(i+1)+'=model_swin'+str(i+1)+'.reshape(-1,self.timestep,in_channels,self.hight,self.weight)', globals(), self.__dict__)
                    else:
                        exec('model_swin'+str(i+1)+'=model_swin'+str(i+1)+'.reshape(-1,in_channels,self.hight,self.weight)', globals(), self.__dict__)
                elif self.model_list[i][0] == 'cbam' :
                    if i==0:
                        exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(x)', globals(), self.__dict__)
                        exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(x)', globals(), self.__dict__)
                        exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                        exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                        exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                        exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                        exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                        exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                        exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*x', globals(), self.__dict__)
                        if self.timestep!=None:
                            exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                            exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        else:
                            exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                            exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                    else:
                        if self.model_list[i-1][0]=='cov' :
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_conv'+str(i), globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='resnet' :
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_add'+str(i), globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='vit' :
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_vit'+str(i), globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='swin' :
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_swin'+str(i), globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='cbam' :
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_cbam_out'+str(i)+'_2', globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_pool'+str(i), globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_norm'+str(i), globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='activation' :
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_act'+str(i), globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='dropout' :
                            exec('model_cbam_max_pool'+str(i+1)+'=self.cbam_max_pool'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_ave_pool'+str(i+1)+'=self.cbam_ave_pool'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_max=self.cbam_conv'+str(i+1)+'_1(model_cbam_max_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_1_ave=self.cbam_conv'+str(i+1)+'_1(model_cbam_ave_pool'+str(i+1)+')', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_max=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_act'+str(i+1)+'_1_ave=self.cbam_act'+str(i+1)+'_1(model_cbam_conv'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_max=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_max)', globals(), self.__dict__)
                            exec('model_cbam_conv'+str(i+1)+'_2_ave=self.cbam_conv'+str(i+1)+'_2(model_cbam_act'+str(i+1)+'_1_ave)', globals(), self.__dict__)
                            exec('model_cbam_out'+str(i+1)+'_1=(self.cbam_act'+str(i+1)+'_2(model_cbam_conv'+str(i+1)+'_2_max+model_cbam_conv'+str(i+1)+'_2_ave))*model_drop'+str(i), globals(), self.__dict__)
                            if self.timestep!=None:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=2, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=2))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                            else:
                                exec('model_cbam_sa'+str(i+1)+'_max=torch.max(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)[0]', globals(), self.__dict__)
                                exec('model_cbam_sa'+str(i+1)+'_ave=torch.mean(model_cbam_out'+str(i+1)+'_1, dim=1, keepdim=True)', globals(), self.__dict__)
                                exec('model_cbam_out'+str(i+1)+'_2=(self.cbam_act'+str(i+1)+'_2(self.cbam_conv'+str(i+1)+'_3(torch.cat([model_cbam_sa'+str(i+1)+'_max,model_cbam_sa'+str(i+1)+'_ave],dim=1))))*model_cbam_out'+str(i+1)+'_1', globals(), self.__dict__)
                elif self.model_list[i][0] == 'maxpooling' or self.model_list[i][0] == 'avepooling':
                    if i==0:
                        exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if self.model_list[i-1][0]=='cov' :
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='resnet' :
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='vit' :
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='swin' :
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='cbam' :
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='activation' :
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                        elif self.model_list[i-1][0]=='dropout' :
                            exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            if ifmulti_scale=='yes':
                                exec('model_pool_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                                try:
                                    if self.timestep!=None:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                                    else:
                                        exec('model_conc=self.conc'+str(i+1)+'((model_conc,model_pool_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                                except:
                                    exec('model_conc=model_pool_fla'+str(i+1), globals(), self.__dict__)
                elif self.model_list[i][0] == 'batchnormalization' or self.model_list[i][0] == 'layernormalization':
                    if i ==0:
                        if i==len(self.model_list)-1:
                            outputs=eval('self.norm'+str(i+1)+'(x)', globals(), self.__dict__)
                        else:
                            exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if i==len(self.model_list)-1:
                            if self.model_list[i-1][0]=='cov' :
                                outputs=eval('self.norm'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='resnet' :
                                outputs=eval('self.norm'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='vit' :
                                outputs=eval('self.norm'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='swin' :
                                outputs=eval('self.norm'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='cbam' :
                                outputs=eval('self.norm'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='transformer' :
                                outputs=eval('self.norm'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='tcn' :
                                outputs=eval('self.norm'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='lstm' :
                                outputs=eval('self.norm'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='rnn' :
                                outputs=eval('self.norm'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='gru' :
                                outputs=eval('self.norm'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                                outputs=eval('self.norm'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                                outputs=eval('self.norm'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                outputs=eval('self.norm'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout' :
                                outputs=eval('self.norm'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='fc':
                                outputs=eval('self.norm'+str(i+1)+'(model_fc'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='flatten':
                                outputs=eval('self.norm'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                        else:
                            if self.model_list[i-1][0]=='cov' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='resnet' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='vit' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='swin' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='cbam' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='transformer' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='tcn' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                            if self.model_list[i-1][0]=='lstm' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='rnn' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='gru' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout' :
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='fc':
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_fc'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='flatten':
                                exec('model_norm'+str(i+1)+'=self.norm'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                elif self.model_list[i][0] == 'activation':
                    if i==0:
                        if i==len(self.model_list)-1:
                            outputs=eval('self.act'+str(i+1)+'(x)', globals(), self.__dict__)
                        else:
                            exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if i==len(self.model_list)-1:
                            if self.model_list[i-1][0]=='cov' :
                                outputs=eval('self.act'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='resnet' :
                                outputs=eval('self.act'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='vit' :
                                outputs=eval('self.act'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='swin' :
                                outputs=eval('self.act'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='cbam' :
                                outputs=eval('self.act'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='transformer' :
                                outputs=eval('self.act'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='tcn' :
                                outputs=eval('self.act'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='lstm' :
                                outputs=eval('self.act'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='rnn' :
                                outputs=eval('self.act'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='gru' :
                                outputs=eval('self.act'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                                outputs=eval('self.act'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                                outputs=eval('self.act'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                outputs=eval('self.act'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout' :
                                outputs=eval('self.act'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='fc':
                                outputs=eval('self.act'+str(i+1)+'(model_fc'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='flatten':
                                outputs=eval('self.act'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                        else:
                            if self.model_list[i-1][0]=='cov':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='resnet':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='vit':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='swin':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='cbam':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='transformer':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='tcn':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='lstm':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='rnn':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='gru':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout' :
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='fc':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_fc'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='flatten':
                                exec('model_act'+str(i+1)+'=self.act'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                elif self.model_list[i][0] == 'flatten':
                    if i ==0:
                        if i==len(self.model_list)-1:
                            outputs=eval('self.fla'+str(i+1)+'(x)', globals(), self.__dict__)
                        else:
                            exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if i==len(self.model_list)-1:
                            if self.model_list[i-1][0]=='cov' :
                                outputs=eval('self.fla'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='resnet' :
                               outputs=eval('self.fla'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='vit' :
                                outputs=eval('self.fla'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='swin' :
                                outputs=eval('self.fla'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='cbam' :
                                outputs=eval('self.fla'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                                outputs=eval('self.fla'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                                outputs=eval('self.fla'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                outputs=eval('self.fla'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout':
                                outputs=eval('self.fla'+str(i+1)+'(model_dropout'+str(i)+')', globals(), self.__dict__)
                        else:
                            if self.model_list[i-1][0]=='cov' :
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='resnet' :
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='vit' :
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='swin' :
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='cbam' :
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout':
                                exec('model_fla'+str(i+1)+'=self.fla'+str(i+1)+'(model_dropout'+str(i)+')', globals(), self.__dict__)
                        if ifmulti_scale=='yes':
                            if timestep !=None:
                                exec('model_fla'+str(i+1)+'=self.conc'+str(i+1)+'((model_conc,model_fla'+str(i+1)+'),axis=2)', globals(), self.__dict__)
                            else:
                                exec('model_fla'+str(i+1)+'=self.conc'+str(i+1)+'((model_conc,model_fla'+str(i+1)+'),axis=1)', globals(), self.__dict__)
                elif self.model_list[i][0] =='fc':
                    if i==0:
                        if i==len(self.model_list)-1:
                            outputs=eval('self.fc'+str(i+1)+'(x)', globals(), self.__dict__)
                        else:
                            exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if i==len(self.model_list)-1:
                            if self.model_list[i-1][0]=='batchnormalization' :
                                outputs=eval('self.fc'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='transformer' :
                                outputs=eval('self.fc'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='tcn' :
                                outputs=eval('self.fc'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='lstm' :
                                outputs=eval('self.fc'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='rnn' :
                                outputs=eval('self.fc'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='gru' :
                                outputs=eval('self.fc'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                outputs=eval('self.fc'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout':
                                outputs=eval('self.fc'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='fc':
                                outputs=eval('self.fc'+str(i+1)+'(model_fc'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='flatten':
                                outputs=eval('self.fc'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                        else:
                            if self.model_list[i-1][0]=='batchnormalization' :
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='transformer' :
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='tcn' :
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='lstm' :
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='rnn' :
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='gru' :
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout':
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='fc':
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_fc'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='flatten':
                                exec('model_fc'+str(i+1)+'=self.fc'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                elif self.model_list[i][0] == 'dropout':
                    if i==0:
                        if i==len(self.model_list)-1:
                            outputs=eval('self.drop'+str(i+1)+'(x)', globals(), self.__dict__)
                        else:
                            exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(x)', globals(), self.__dict__)
                    else:
                        if i==len(self.model_list)-1:
                            if self.model_list[i-1][0]=='cov' :
                                outputs=eval('self.drop'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='resnet' :
                                outputs=eval('self.drop'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='vit' :
                                outputs=eval('self.drop'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='swin' :
                                outputs=eval('self.drop'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='cbam' :
                                outputs=eval('self.drop'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='transformer' :
                                outputs=eval('self.drop'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='tcn' :
                                outputs=eval('self.drop'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='lstm' :
                                outputs=eval('self.drop'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='rnn' :
                                outputs=eval('self.drop'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='gru' :
                                outputs=eval('self.drop'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                                outputs=eval('self.drop'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                                outputs=eval('self.drop'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                outputs=eval('self.drop'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout' :
                                outputs=eval('self.drop'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='fc':
                                outputs=eval('self.drop'+str(i+1)+'(model_fc'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='flatten':
                                outputs=eval('self.drop'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                        else:
                            if self.model_list[i-1][0]=='cov':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_conv'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='resnet':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_add'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='vit':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_vit'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='swin':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_swin'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='cbam':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_cbam_out'+str(i)+'_2)', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='transformer':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='tcn':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='lstm':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='rnn':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='gru':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='maxpooling' or self.model_list[i-1][0]=='avepooling':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='batchnormalization' or self.model_list[i-1][0]=='layernormalization':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='activation':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='dropout' :
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='fc':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_fc'+str(i)+')', globals(), self.__dict__)
                            elif self.model_list[i-1][0]=='flatten':
                                exec('model_drop'+str(i+1)+'=self.drop'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                elif self.model_list[i][0] == 'transformer':
                    if i ==0:
                        exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                        exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',x)', globals(), self.__dict__)
                        exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_transformer'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_transformer'+str(i+1)+'=model_transformer'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                    else:
                        if model_list[i-1][0] == 'flatten':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_fla'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'transformer':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_transformer'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'tcn':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_tcn'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'lstm':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_lstm'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'rnn':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_rnn'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'gru':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_gru'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='batchnormalization' or model_list[i-1][0]=='layernormalization':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_norm'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='activation':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_act'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='dropout':
                            exec('model_embedding'+str(i+1)+'=self.embedding'+str(i+1)+'(position)', globals(), self.__dict__)
                            exec('model_transformer_add'+str(i+1)+'=self.transformer_add'+str(i+1)+'(model_embedding'+str(i+1)+',model_drop'+str(i)+')', globals(), self.__dict__)
                            exec('model_transformer'+str(i+1)+'=self.transformer'+str(i+1)+'(model_transformer_add'+str(i+1)+')', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_transformer'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_transformer'+str(i+1)+'=model_transformer'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'tcn':
                    if i ==0:
                        exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(x)', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_tcn'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_tcn'+str(i+1)+'=model_tcn'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                    else:
                        if model_list[i-1][0] == 'flatten':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'transformer':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'tcn':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'lstm':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'rnn':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'gru':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='batchnormalization' or model_list[i-1][0]=='layernormalization':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='activation':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='dropout':
                            exec('model_tcn'+str(i+1)+'=self.tcn'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_tcn'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_tcn'+str(i+1)+'=model_tcn'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'lstm':
                    if i ==0:
                        exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(x)', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_lstm'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_lstm'+str(i+1)+'=model_lstm'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                    else:
                        if model_list[i-1][0] == 'flatten':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'transformer':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'tcn':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'lstm':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'rnn':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'gru':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='batchnormalization' or model_list[i-1][0]=='layernormalization':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='activation':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='dropout':
                            exec('model_lstm'+str(i+1)+',(h_0,c_0)=self.lstm'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_lstm'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_lstm'+str(i+1)+'=model_lstm'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'rnn':
                    if i ==0:
                        exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(x)', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_rnn'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_rnn'+str(i+1)+'=model_rnn'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                    else:
                        if model_list[i-1][0] == 'flatten':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'transformer':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'tcn':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'lstm':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'rnn':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'gru':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='batchnormalization' or model_list[i-1][0]=='layernormalization':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='activation':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='dropout':
                            exec('model_rnn'+str(i+1)+',h_0=self.rnn'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_rnn'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_rnn'+str(i+1)+'=model_rnn'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                elif self.model_list[i][0] == 'gru':
                    if i ==0:
                        exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(x)', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_gru'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_gru'+str(i+1)+'=model_gru'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                    else:
                        if model_list[i-1][0] == 'flatten':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_fla'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'transformer':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_transformer'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'tcn':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_tcn'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'lstm':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_lstm'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'rnn':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_rnn'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0] == 'gru':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_gru'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='batchnormalization' or model_list[i-1][0]=='layernormalization':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_norm'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='activation':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_act'+str(i)+')', globals(), self.__dict__)
                        elif model_list[i-1][0]=='dropout':
                            exec('model_gru'+str(i+1)+',h_0=self.gru'+str(i+1)+'(model_drop'+str(i)+')', globals(), self.__dict__)
                        if is_last_occurrence(model_list[i][0], model_list,i):
                            if i==len(self.model_list)-1:
                                outputs=eval('model_gru'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
                            else:
                                exec('model_gru'+str(i+1)+'=model_gru'+str(i+1)+'[:,-1,:]', globals(), self.__dict__)
            return outputs
    if k_fold!=None: 
        models=[]
        for i in range(k_fold):
            models.append(Model(model_list,timestep,dilations,ifmulti_scale,num_channels,trainx[0:2],cov_padding,cov_strides,pooling_strides,dropout,residual_deep,residual_activation,residual_norm,vit_patch_size,vit_dim_feedforward,vit_deep,vit_num_heads,swin_deep,swin_num_heads,cbam_reduction,transformer_encoder_deep,transformer_num_heads,transformer_dim_feedforward,tcn_kernel_size,nlp_units,nlp_num_layers,activation))
    else:
        model=Model(model_list,timestep,dilations,ifmulti_scale,num_channels,trainx[0:2],cov_padding,cov_strides,pooling_strides,dropout,residual_deep,residual_activation,residual_norm,vit_patch_size,vit_dim_feedforward,vit_deep,vit_num_heads,swin_deep,swin_num_heads,cbam_reduction,transformer_encoder_deep,transformer_num_heads,transformer_dim_feedforward,tcn_kernel_size,nlp_units,nlp_num_layers,activation)
    if if_best_mode!='no':
        if k_fold!=None:
            for i in range(k_fold):
                models[i].load_state_dict(torch.load(modelpath+'_'+str(i+1)+'.pth', map_location=devices))
        else:
            model.load_state_dict(torch.load(modelpath+'.pth', map_location=devices))
    if k_fold!=None:
        for i in range(k_fold):
            models[i].to(devices)
    else:
        model.to(devices)
    if k_fold!=None:
        opts=[]
        for i in range(k_fold):
            if optimizer == 'SGD':
                opts.append(SGD(models[i].parameters(), lr=learning_rate))
            elif optimizer == 'Adam':
                opts.append(Adam(models[i].parameters(), lr=learning_rate))
            elif optimizer == 'Nadam':
                opts.append(NAdam(models[i].parameters(), lr=learning_rate))
    else:
        if optimizer == 'SGD':
            opt = SGD(model.parameters(), lr=learning_rate)
        elif optimizer == 'Adam':
            opt = Adam(model.parameters(), lr=learning_rate)
        elif optimizer == 'Nadam':
            opt = NAdam(model.parameters(), lr=learning_rate)
    if if_best_mode!='no':
        if k_fold!=None:
            for i in range(k_fold):
                opts[i].load_state_dict(torch.load(modelpath+'_'+str(i+1)+'_opt.pth', map_location=devices))
        else:
            opt.load_state_dict(torch.load(modelpath+'_opt.pth', map_location=devices))
    if if_print_model=='yes':
        if k_fold!=None:
            print(models[0])
        else:
            print(model)
    if ifrandom_split!='just_model':
        trainx=np.nan_to_num(trainx,nan=0)
        testx=np.nan_to_num(testx,nan=0)
        if ifrandom_split!='all_test':
            if valid_size!=None or k_fold !=None:
                if k_fold!=None:
                    kf = KFold(n_splits=k_fold, shuffle=True, random_state=25)
                    for fold_no, (train_idx, val_idx) in enumerate(kf.split(trainx, trainy)):
                        X_train_fold, y_train_fold,position_train_fold = trainx[train_idx], trainy[train_idx],train_position[train_idx]
                        X_val_fold, y_val_fold,position_val_fold = trainx[val_idx], trainy[val_idx],train_position[val_idx]
                        if fold_no==0:
                            train_loss=np.zeros((k_fold,int(np.ceil(X_train_fold.shape[0]/batch_size))))
                            train_metric=np.zeros((k_fold,int(np.ceil(X_train_fold.shape[0]/batch_size))))
                            test_loss=np.zeros((k_fold,int(np.ceil(X_val_fold.shape[0]/batch_size))))
                            test_metric=np.zeros((k_fold,int(np.ceil(X_val_fold.shape[0]/batch_size))))
                        if if_early_stopping!=None:
                            early_stopping = EarlyStopping(patience=if_early_stopping)
                        for i in range(epochs):
                            start = datetime.datetime.now()
                            for j in range(int(np.ceil(X_train_fold.shape[0]/batch_size))):
                                if j == int(X_train_fold.shape[0]/batch_size) :
                                    train_output = models[fold_no](torch.tensor(X_train_fold[j*batch_size:],dtype=torch.float32,device=devices),torch.tensor(position_train_fold[j*batch_size:],dtype=torch.int64,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.float32,device=devices)))
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.float32,device=devices)))
                                else:
                                    train_output = models[fold_no](torch.tensor(X_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(position_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.int64,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                train_loss[fold_no,j]=np.array(train_losss.item())
                                train_metric[fold_no,j]=np.array(train_metrics.item())
                                opts[fold_no].zero_grad()
                                train_losss.backward(retain_graph=True)
                                opts[fold_no].step()
                            with torch.no_grad():
                                for k in range(int(np.ceil(X_val_fold.shape[0]/batch_size))):
                                    if k == int(X_val_fold.shape[0]/batch_size):
                                        test_output = models[fold_no](torch.tensor(X_val_fold[k*batch_size:],dtype=torch.float32,device=devices),torch.tensor(position_val_fold[k*batch_size:],dtype=torch.int64,device=devices))
                                        if task_mode=='multi_classify':
                                            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                                test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                            elif loss_function=='NLLLoss':
                                                test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                                test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                            elif metrics=='CrossEntropyLoss':
                                                test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                            elif metrics=='NLLLoss':
                                                test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        else:
                                            test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.float32,device=devices)))
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.float32,device=devices)))
                                    else:
                                        test_output = models[fold_no](torch.tensor(X_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(position_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.int64,device=devices))
                                        if task_mode=='multi_classify':
                                            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                                test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                            elif loss_function=='NLLLoss':
                                                test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                                test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                            elif metrics=='CrossEntropyLoss':
                                                test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                            elif metrics=='NLLLoss':
                                                test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        else:
                                            test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                    test_loss[fold_no,k]=np.array(test_losss.item())
                                    test_metric[fold_no,k]=np.array(test_metrics.item())
                            if if_early_stopping!=None:
                                early_stopping(np.nanmean(test_loss[fold_no,:]), models[fold_no])
                                if early_stopping.early_stop:
                                    early_stopping.load_best_checkpoint(models[fold_no])
                                    break
                            end = datetime.datetime.now()
                            print('第',i+1,'次训练loss:',np.nanmean(train_loss[fold_no,:]),'，metric:',np.nanmean(train_metric[fold_no,:]),'  第',i+1,'次测试loss:',np.nanmean(test_loss[fold_no,:]),'，metric：',np.nanmean(test_metric[fold_no,:]),'训练用时:',end - start)
                else:
                    if if_early_stopping!=None:
                        early_stopping = EarlyStopping(patience=if_early_stopping)
                    for i in range(epochs):
                        start = datetime.datetime.now()
                        if i==0:
                            if ifrandom_split=='yes':
                                trainy,validy,trainx,validx,train_position,valid_position = train_test_split(trainy,trainx,train_position,test_size=valid_size/(1-test_size),random_state=25)
                            else:
                                index=int((1-valid_size/(1-test_size))*trainy.shape[0])
                                validy=trainy[index:]
                                trainy=trainy[:index]
                                validx=trainx[index:]
                                trainx=trainx[:index]
                                valid_position=train_position[index:]
                                train_position=train_position[:index]
                        train_loss=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                        train_metric=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                        test_loss=np.zeros((int(np.ceil(validx.shape[0]/batch_size))))
                        test_metric=np.zeros((int(np.ceil(validx.shape[0]/batch_size))))
                        for j in range(int(np.ceil(trainx.shape[0]/batch_size))):
                            if j == int(trainx.shape[0]/batch_size) :
                                train_output = model(torch.tensor(trainx[j*batch_size:],dtype=torch.float32,device=devices),torch.tensor(train_position[j*batch_size:],dtype=torch.int64,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                            else:
                                train_output = model(torch.tensor(trainx[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(train_position[j*batch_size:(j+1)*batch_size],dtype=torch.int64,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                            train_loss[j]=np.array(train_losss.item())
                            train_metric[j]=np.array(train_metrics.item())
                            opt.zero_grad()
                            train_losss.backward(retain_graph=True)
                            opt.step() 
                        with torch.no_grad():
                            for k in range(int(np.ceil(validx.shape[0]/batch_size))):
                                if k == int(validx.shape[0]/batch_size) :
                                    test_output = model(torch.tensor(validx[k*batch_size:],dtype=torch.float32,device=devices),torch.tensor(valid_position[k*batch_size:],dtype=torch.int64,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.float32,device=devices)))
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.float32,device=devices)))
                                else:
                                    test_output = model(torch.tensor(validx[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(valid_position[k*batch_size:(k+1)*batch_size],dtype=torch.int64,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                test_loss[k]=np.array(test_losss.item())
                                test_metric[k]=np.array(test_metrics.item())
                        if if_early_stopping!=None:
                            early_stopping(np.nanmean(test_loss), model)
                            if early_stopping.early_stop:
                                early_stopping.load_best_checkpoint(model)
                                break
                        end = datetime.datetime.now()
                        print('第',i+1,'次训练loss:',np.nanmean(train_loss),'，metric:',np.nanmean(train_metric),'  第',i+1,'次测试loss:',np.nanmean(test_loss),'，metric：',np.nanmean(test_metric),'训练用时:',end - start)
            else:
                if if_early_stopping!=None:
                    early_stopping = EarlyStopping(patience=if_early_stopping)
                for i in range(epochs):
                    start = datetime.datetime.now()
                    train_loss=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                    train_metric=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                    test_loss=np.zeros((int(np.ceil(testx.shape[0]/batch_size))))
                    test_metric=np.zeros((int(np.ceil(testx.shape[0]/batch_size))))
                    for j in range(int(np.ceil(trainx.shape[0]/batch_size))):
                        if j == int(trainx.shape[0]/batch_size):
                            train_output = model(torch.tensor(trainx[j*batch_size:],dtype=torch.float32,device=devices),torch.tensor(train_position[j*batch_size:],dtype=torch.int64,device=devices))
                            if task_mode=='multi_classify':
                                if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif loss_function=='NLLLoss':
                                    train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                    train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='CrossEntropyLoss':
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='NLLLoss':
                                    train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                            else:
                                train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                                train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                        else:
                            train_output = model(torch.tensor(trainx[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(train_position[j*batch_size:(j+1)*batch_size],dtype=torch.int64,device=devices))
                            if task_mode=='multi_classify':
                                if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif loss_function=='NLLLoss':
                                    train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                    train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='CrossEntropyLoss':
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='NLLLoss':
                                    train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                            else:
                                train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                        train_loss[j]=np.array(train_losss.item())
                        train_metric[j]=np.array(train_metrics.item())
                        opt.zero_grad()
                        train_losss.backward(retain_graph=True)
                        opt.step()
                    with torch.no_grad():
                        for k in range(int(np.ceil(testx.shape[0]/batch_size))):
                            if k == int(testx.shape[0]/batch_size) :
                                test_output = model(torch.tensor(testx[k*batch_size:],dtype=torch.float32,device=devices),torch.tensor(test_position[k*batch_size:],dtype=torch.int64,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.float32,device=devices)))
                                    test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.float32,device=devices)))
                            else:
                                test_output = model(torch.tensor(testx[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(test_position[k*batch_size:(k+1)*batch_size],dtype=torch.int64,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                    test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                            test_loss[k]=np.array(test_losss.item())
                            test_metric[k]=np.array(test_metrics.item())
                    if if_early_stopping!=None:
                        early_stopping(np.nanmean(test_loss), model)
                        if early_stopping.early_stop:
                            early_stopping.load_best_checkpoint(model)
                            break
                    end = datetime.datetime.now()
                    print('第',i+1,'次训练loss:',np.nanmean(train_loss),'，metric:',np.nanmean(train_metric),'  第',i+1,'次测试loss:',np.nanmean(test_loss),'，metric：',np.nanmean(test_metric),'训练用时:',end - start)
        if k_fold!=None:
            if task_mode=='multi_classify':
                predicty=np.zeros((k_fold,testy.shape[0],int(np.max(vy))+1))
            else:
                predicty=np.zeros((k_fold,testy.shape[0],testy.shape[1]))
        else:
            if task_mode=='multi_classify':
                predicty=np.zeros((testy.shape[0],int(np.max(vy))+1))
            else:
                predicty=np.zeros((testy.shape[0],testy.shape[1]))
        if k_fold!=None:
            for i in range(k_fold):
                for k in range(int(np.ceil(testx.shape[0]/batch_size))):
                    if k == int(testx.shape[0]/batch_size) :
                        predicty_batch = models[i](torch.tensor(testx[k*batch_size:],dtype=torch.float32,device=devices),torch.tensor(test_position[k*batch_size:],dtype=torch.int64,device=devices))
                        predicty[i,k*batch_size:]=predicty_batch.cpu().detach().numpy()
                    else:
                        predicty_batch = models[i](torch.tensor(testx[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(test_position[k*batch_size:(k+1)*batch_size],dtype=torch.int64,device=devices))
                        predicty[i,k*batch_size:(k+1)*batch_size]=predicty_batch.cpu().detach().numpy()
            predicty=np.nanmean(predicty,axis=0)
        else:
            for k in range(int(np.ceil(testx.shape[0]/batch_size))):
                if k == int(testx.shape[0]/batch_size) :
                    predicty_batch = model(torch.tensor(testx[k*batch_size:],dtype=torch.float32,device=devices),torch.tensor(test_position[k*batch_size:],dtype=torch.int64,device=devices))
                    predicty[k*batch_size:]=predicty_batch.cpu().detach().numpy()
                else:
                    predicty_batch = model(torch.tensor(testx[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(test_position[k*batch_size:(k+1)*batch_size],dtype=torch.int64,device=devices))
                    predicty[k*batch_size:(k+1)*batch_size]=predicty_batch.cpu().detach().numpy()
        predicty = np.nan_to_num(predicty,nan=0)
        if task_mode=='regression':
            r=np.zeros((testy.shape[1]))
            p=np.zeros((testy.shape[1]))
            for i in range(testy.shape[1]):
                r[i],p[i] = pearsonr(predicty[:,i],testy[:,i])
                r=np.nan_to_num(r,nan=0)
        elif task_mode=='binary_classify':
            accuracy=np.zeros((testy.shape[1]))
            recall=np.zeros((testy.shape[1]))
            precision=np.zeros((testy.shape[1]))
            f1=np.zeros((testy.shape[1]))
            for i in range(predicty.shape[1]):
                predicty[:,i]=[int(round(predicty[j,i],0)) for j in range(predicty.shape[0])]
            r=np.zeros((testy.shape[1]))
            for i in range(testy.shape[1]):
                if metrics=='Recall':
                    r[i]=recall_score(testy[:,i], predicty[:,i])
                elif metrics=='Precision':
                    r[i]=precision_score(testy[:,i], predicty[:,i])
                else:
                    r[i]=accuracy_score(testy[:,i], predicty[:,i])
                recall[i]=recall_score(testy[:,i], predicty[:,i])
                precision[i]=precision_score(testy[:,i], predicty[:,i])
                accuracy[i]=accuracy_score(testy[:,i], predicty[:,i])
                f1[i]=f1_score(testy[:,i], predicty[:,i])
            p=0
        elif task_mode=='multi_classify':
            r=np.zeros((testy.shape[1]))
            for i in range(testy.shape[1]):
                r[i]=accuracy_score(testy[:,i], np.argmax(predicty,axis=1))
            p=0
        if ifmute == 'no':
            if task_mode=='regression':
                print('相关系数',np.nanmean(r))
            elif task_mode=='binary_classify':
                print('召回率+精确率',np.nanmean(f1),'准确率',np.nanmean(accuracy),'召回率',np.nanmean(recall),'精确率',np.nanmean(precision))
            elif task_mode=='multi_classify':
                print('准确率',np.nanmean(r))
        current_testx_data = None
        current_test_position_data = None
        sample_x_shape = None
        sample_pos_shape = None
        num_model_outputs = None
        combined_input_for_shap_data = None
        
        if (ifheatmap == 'yes' and testx.ndim > 3) or ifweight == 'yes' or ifweight == 'shap':
            current_testx_data = testx
            current_test_position_data = test_position
            if testx.shape[0] >= 100:
                index = np.random.randint(0, testx.shape[0], size=100)
                current_testx_data = testx[index, ]
                current_test_position_data = test_position[index, ]
        
            sample_x_shape = current_testx_data.shape[1:]
            sample_pos_shape = current_test_position_data.shape[1:]
        
            flat_current_testx_tensor = torch.tensor(current_testx_data, dtype=torch.float32, device=devices).flatten(start_dim=1)
            flat_current_test_position_tensor = torch.tensor(current_test_position_data, dtype=torch.float32, device=devices).flatten(start_dim=1)
            combined_input_for_shap_data = torch.cat((flat_current_testx_tensor, flat_current_test_position_tensor), dim=1)
        
            num_model_outputs = testy.shape[1]
        
        has_transformer_layer = find_string_in_nested_list('transformer', model_list)
        has_rnn_like_layer = find_string_in_nested_list('lstm', model_list) or \
                             find_string_in_nested_list('gru', model_list) or \
                             find_string_in_nested_list('rnn', model_list) or \
                             find_string_in_nested_list('tcn', model_list)
        has_vit_swin_layer = find_string_in_nested_list('vit', model_list) or \
                             find_string_in_nested_list('swin', model_list)
        
        use_gradient_explainer = has_transformer_layer or has_rnn_like_layer or has_vit_swin_layer
        disable_cudnn_for_rnn_tcn = has_rnn_like_layer
        
        def _compute_shap_values_for_model(model_instance):
            wrapped_model_combined = ShapWrapperCombinedInput(model_instance, sample_x_shape, sample_pos_shape)
        
            if use_gradient_explainer:
                explainer = shap.GradientExplainer(wrapped_model_combined, combined_input_for_shap_data)
            else:
                explainer = shap.DeepExplainer(wrapped_model_combined, combined_input_for_shap_data)
            
            raw_shap_values = explainer.shap_values(combined_input_for_shap_data)
        
            if isinstance(raw_shap_values, np.ndarray) and raw_shap_values.ndim == 3 and raw_shap_values.shape[2] == num_model_outputs:
                raw_shap_values = [raw_shap_values[:, :, i] for i in range(num_model_outputs)]
            elif not isinstance(raw_shap_values, list) and num_model_outputs > 1:
                total_flat_size_per_output = wrapped_model_combined.x_flat_size + wrapped_model_combined.pos_flat_size
                if raw_shap_values.shape[1] == num_model_outputs * total_flat_size_per_output:
                    split_shap_values = []
                    for i in range(num_model_outputs):
                        start_idx = i * total_flat_size_per_output
                        end_idx = (i + 1) * total_flat_size_per_output
                        split_shap_values.append(raw_shap_values[:, start_idx:end_idx])
                    raw_shap_values = split_shap_values
                else:
                    raw_shap_values = [raw_shap_values]
            elif not isinstance(raw_shap_values, list) and num_model_outputs == 1:
                raw_shap_values = [raw_shap_values]
        
            current_model_shap_x_outputs = []
            if isinstance(raw_shap_values, list):
                for output_shap_array in raw_shap_values:
                    shap_x_flat = output_shap_array[:, :wrapped_model_combined.x_flat_size]
                    shap_x_reshaped = shap_x_flat.reshape(output_shap_array.shape[0], *wrapped_model_combined.x_sample_shape)
                    current_model_shap_x_outputs.append(np.abs(shap_x_reshaped))
                return np.stack(current_model_shap_x_outputs, axis=0)
            else:
                shap_x_flat = raw_shap_values[:, :wrapped_model_combined.x_flat_size]
                shap_x_reshaped = shap_x_flat.reshape(raw_shap_values.shape[0], *wrapped_model_combined.x_sample_shape)
                return np.abs(shap_x_reshaped)
        
        shap_results = None  
        
        should_compute_shap = (ifheatmap == 'yes' and testx.ndim > 3) or (ifweight in ['yes', 'shap'])
        
        if should_compute_shap:
            if k_fold != None:
                all_models_shap_x_results = []
                for model_k_fold in models:
                    model_k_fold.eval()
                    if disable_cudnn_for_rnn_tcn:
                        with torch.backends.cudnn.flags(enabled=False):
                            all_models_shap_x_results.append(_compute_shap_values_for_model(model_k_fold))
                    else:
                        all_models_shap_x_results.append(_compute_shap_values_for_model(model_k_fold))
                
                shap_results = np.nanmean(np.stack(all_models_shap_x_results, axis=0), axis=0)
            else:
                model.eval()
                if disable_cudnn_for_rnn_tcn:
                    with torch.backends.cudnn.flags(enabled=False):
                        shap_results = _compute_shap_values_for_model(model)
                else:
                    shap_results = _compute_shap_values_for_model(model)
        
        if ifheatmap == 'yes' and testx.ndim > 3:
            heatmap_more = shap_results
            
            if timestep is not None:
                heatmap = np.nanmean(heatmap_more, axis=(1, 2))
            else:
                heatmap = np.nanmean(heatmap_more, axis=1)
        
        if ifweight == 'yes' or ifweight == 'shap':
            weights_more = shap_results 
            
            if timestep is not None:
                feature_axis_in_sample_x_shape = 1
            else:
                feature_axis_in_sample_x_shape = 0
        
            feature_axis_in_weights_more = 2 + feature_axis_in_sample_x_shape
        
            axes_to_average = [1]
        
            for ax_idx in range(len(sample_x_shape)):
                current_axis_in_weights_more = 2 + ax_idx
                if current_axis_in_weights_more != feature_axis_in_weights_more:
                    axes_to_average.append(current_axis_in_weights_more)
            
            weights = np.nanmean(weights_more, axis=tuple(axes_to_average))
        
            if weights.ndim == 1:
                weights = weights.reshape(1, -1)
        
            if timestep != None:
                input_channel_for_weights = trainx.shape[2]
            else:
                input_channel_for_weights = trainx.shape[1]
        
            if weights.shape[0] == num_model_outputs and weights.shape[1] == input_channel_for_weights:
                weight_for_loop = weights.transpose(1, 0)
            elif weights.shape[0] == 1 and weights.shape[1] == input_channel_for_weights:
                weight_for_loop = weights.reshape(-1, 1)
            else:
                weight_for_loop = np.zeros((input_channel_for_weights, num_model_outputs))
            
            weights_final_output = np.zeros((num_model_outputs, input_channel_for_weights))
        
            for i in range(num_model_outputs):
                for j in range(input_channel_for_weights):
                    if weight_for_loop.shape[0] > j and weight_for_loop.shape[1] > i:
                        sum_for_norm = np.nansum(weight_for_loop[:, i])
                        if sum_for_norm != 0:
                            contribution_val = (weight_for_loop[j, i] / sum_for_norm) * 100
                        else:
                            contribution_val = 0.0
                        weights_final_output[i, j] = contribution_val
                        print('预报因子', j+1, '对预报值', i+1, '的贡献：', np.array(weights_final_output[i, j]), '％')
                    else:
                        pass 
                print('\n')
            
            weights = weights_final_output
    
    
        elif ifweight=='oob':
            if task_mode=='multi_classify':
                weights=np.zeros((int(np.max(vy))+1,input_channel))
                weight_more=np.zeros((int(np.max(vy))+1,input_channel))
            else:
                weights=np.zeros((testy.shape[1],input_channel))
                weight_more=np.zeros((testy.shape[1],input_channel))
            for i in tqdm(range(testy.shape[1])):
                for j in range(input_channel):
                    testx_new=copy.deepcopy(testx)
                    weight=[]
                    for k in range(10):
                        per=np.random.permutation(testx.shape[0])
                        testx_shuffle=testx[per,:,j]
                        testx_new[:,:,j]=testx_shuffle
                        if k_fold!=None:
                            if task_mode=='multi_classify':
                                predicty_new=np.zeros((k_fold,testy.shape[0],np.max(vy)+1))
                            else:
                                predicty_new=np.zeros((k_fold,testy.shape[0],testy.shape[1]))
                            for n in range(k_fold):
                                for l in range(int(np.ceil(testx.shape[0]/batch_size))):
                                    if l == int(testx.shape[0]/batch_size):
                                        predicty_new_batch = models[n](torch.tensor(testx_new[l*batch_size:],dtype=torch.float32,device=devices),torch.tensor(test_position[l*batch_size:],dtype=torch.int64,device=devices))
                                        predicty_new[n,l*batch_size:,:]=predicty_new_batch.cpu().detach().numpy()
                                    else:
                                        predicty_new_batch = models[n](torch.tensor(testx_new[l*batch_size:(l+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(test_position[l*batch_size:(l+1)*batch_size],dtype=torch.int64,device=devices))
                                        predicty_new[n,l*batch_size:(l+1)*batch_size,:]=predicty_new_batch.cpu().detach().numpy()
                            predicty_new=np.nanmean(predicty_new,axis=0)
                        else:
                            if task_mode=='multi_classify':
                                predicty_new=np.zeros((testy.shape[0],np.max(vy)+1))
                            else:
                                predicty_new=np.zeros((testy.shape[0],testy.shape[1]))
                            for l in range(int(np.ceil(testx.shape[0]/batch_size))):
                                if l == int(testx.shape[0]/batch_size) :
                                    predicty_new_batch = model(torch.tensor(testx_new[l*batch_size:],dtype=torch.float32,device=devices),torch.tensor(test_position[l*batch_size:],dtype=torch.int64,device=devices))
                                    predicty_new[l*batch_size:,:]=predicty_new_batch.cpu().detach().numpy()
                                else:
                                    predicty_new_batch = model(torch.tensor(testx_new[l*batch_size:(l+1)*batch_size],dtype=torch.float32,device=devices),torch.tensor(test_position[l*batch_size:(l+1)*batch_size],dtype=torch.int64,device=devices))
                                    predicty_new[l*batch_size:(l+1)*batch_size,:]=predicty_new_batch.cpu().detach().numpy()
                        if task_mode=='regression':
                            weight.append(sklearn.metrics.mean_squared_error(testy[:,i],predicty_new[:,i])-sklearn.metrics.mean_squared_error(testy[:,i],predicty[:,i]))
                        elif task_mode=='multi_classify':
                            weight.append(sklearn.metrics.log_loss(testy[:,i],predicty_new[:,:])-sklearn.metrics.log_loss(testy[:,i],predicty[:,:]))
                        else:
                            weight.append(sklearn.metrics.log_loss(testy[:,i],predicty_new[:,i])-sklearn.metrics.log_loss(testy[:,i],predicty[:,i]))
                    weight_more[i,j]=np.nanmean(weight)
            for i in range(testy.shape[1]):
                for j in range(input_channel):
                    weights[i,j]=(weight_more[i,j]/np.nansum(weight_more[i,:]))*100
                    print('预报因子',j+1,'对预报值',i+1,'的贡献：',np.array(weights[i,j]),'％')
                print('\n')
    if ifsave=='yes':
        if k_fold!=None:
            for i in range(k_fold):
                torch.save(models[i].state_dict(),savepath+'_'+str(i+1)+'.pth')
                torch.save(opts[i].state_dict(),savepath+'_'+str(i+1)+'_opt.pth')
        else:
            torch.save(model.state_dict(),savepath+'.pth')
            torch.save(opt.state_dict(),savepath+'_opt.pth')
    if k_fold!=None:    
        return models,predicty,testy,r,p,heatmap,weights
    else:
        return model,predicty,testy,r,p,heatmap,weights